# Сверка Excel vs Datamart за февраль: INN / AGR / Номер договора

Тетрадка считает:
- статистику по уникальным `inn`, `agr_id`, `c_agr_number`;
- пересечения множеств между Excel и витриной техлида;
- примеры расхождений (только в Excel / только в Datamart).

Принятые соответствия полей:
- Excel `ID договора` = Datamart `agr_id`
- Excel `ИНН` = Datamart `inn`
- Excel `Номер договора` = Datamart `c_agr_number`

Период Datamart: только февраль (`trx_month = 2026-02-01`).
Excel уже содержит февральский отчет.

In [ ]:
import re
from decimal import Decimal, InvalidOperation
import pandas as pd
from rail_connectors.connection import connect

pd.options.display.max_columns = None
pd.options.display.width = None
pd.options.display.max_colwidth = None

In [ ]:
# Параметры
excel_path = "/home/jovyan/documents/Equaring/Проверки/отчет_февраль_2026юxlsx"
excel_agr_col = "ID договора"
excel_inn_col = "ИНН"
excel_contract_col = "Номер договора"

month_start = "2026-02-01"  # для Datamart
datamart_table = "sandbox_ai.stg__chesnov_aef__sa_acquiring_merchants"

In [ ]:
# Подключение к Impala (формат без LAKE_USER/LAKE_PASSWORD)
imp = connect(
    to='IMPALA',
    extra_options={'db': 'sandbox_ai'},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params={'user_name': 'Shestopalov-VYur'}
)
imp._init_connection()
print('Impala connection initialized')

In [ ]:
def normalize_id(v):
    """Нормализует ID (в т.ч. scientific notation из Excel) к строке из цифр."""
    if pd.isna(v):
        return None

    s = str(v).strip().replace(" ", "").replace("\xa0", "")
    if s == "" or s.lower() in {"nan", "none", "null"}:
        return None

    s = s.replace(",", ".")

    if re.search(r"[eE][+-]?\d+$", s):
        try:
            s = format(Decimal(s), "f")
        except InvalidOperation:
            return None

    s = re.sub(r"\.0+$", "", s)
    return s if re.fullmatch(r"\d+", s) else None


def normalize_str(v):
    if pd.isna(v):
        return None
    s = str(v).strip()
    if s == "" or s.lower() in {"nan", "none", "null"}:
        return None
    return s

## 1) Загружаем Excel и нормализуем ключевые поля

In [ ]:
df_excel = pd.read_excel(excel_path)
print('excel_rows_total =', len(df_excel))

for col in [excel_agr_col, excel_inn_col, excel_contract_col]:
    if col not in df_excel.columns:
        raise ValueError(f"В Excel не найдена колонка: {col}")

excel_norm = pd.DataFrame({
    "agr_id": df_excel[excel_agr_col].apply(normalize_id),
    "inn": df_excel[excel_inn_col].apply(normalize_id),
    "c_agr_number": df_excel[excel_contract_col].apply(normalize_str),
}).dropna(how='all')

display(excel_norm.head(10))

## 2) Загружаем Datamart только за февраль

In [ ]:
with imp:
    dm = imp.fetch(f"""
        select
            cast(agr_id as string) as agr_id,
            cast(inn as string) as inn,
            cast(c_agr_number as string) as c_agr_number
        from {datamart_table}
        where trx_month = cast('{month_start}' as date)
    """)

dm_norm = pd.DataFrame({
    "agr_id": dm["agr_id"].apply(normalize_id),
    "inn": dm["inn"].apply(normalize_id),
    "c_agr_number": dm["c_agr_number"].apply(normalize_str),
}).dropna(how='all')

print('datamart_rows_feb =', len(dm_norm))
display(dm_norm.head(10))

## 3) Базовая статистика по уникальным сущностям

In [ ]:
excel_stats = {
    "excel_unique_inn": excel_norm["inn"].dropna().nunique(),
    "excel_unique_agr_id": excel_norm["agr_id"].dropna().nunique(),
    "excel_unique_c_agr_number": excel_norm["c_agr_number"].dropna().nunique(),
}

dm_stats = {
    "datamart_unique_inn": dm_norm["inn"].dropna().nunique(),
    "datamart_unique_agr_id": dm_norm["agr_id"].dropna().nunique(),
    "datamart_unique_c_agr_number": dm_norm["c_agr_number"].dropna().nunique(),
}

summary_stats = pd.DataFrame([
    {"metric": k, "value": v} for k, v in {**excel_stats, **dm_stats}.items()
])
display(summary_stats)

## 4) Пересечения по INN / AGR_ID / C_AGR_NUMBER

In [ ]:
set_excel_inn = set(excel_norm["inn"].dropna().tolist())
set_dm_inn = set(dm_norm["inn"].dropna().tolist())

set_excel_agr = set(excel_norm["agr_id"].dropna().tolist())
set_dm_agr = set(dm_norm["agr_id"].dropna().tolist())

set_excel_num = set(excel_norm["c_agr_number"].dropna().tolist())
set_dm_num = set(dm_norm["c_agr_number"].dropna().tolist())

compare_summary = pd.DataFrame([
    {
        "entity": "inn",
        "excel_unique": len(set_excel_inn),
        "datamart_unique": len(set_dm_inn),
        "intersect_cnt": len(set_excel_inn & set_dm_inn),
        "only_excel_cnt": len(set_excel_inn - set_dm_inn),
        "only_datamart_cnt": len(set_dm_inn - set_excel_inn),
    },
    {
        "entity": "agr_id",
        "excel_unique": len(set_excel_agr),
        "datamart_unique": len(set_dm_agr),
        "intersect_cnt": len(set_excel_agr & set_dm_agr),
        "only_excel_cnt": len(set_excel_agr - set_dm_agr),
        "only_datamart_cnt": len(set_dm_agr - set_excel_agr),
    },
    {
        "entity": "c_agr_number",
        "excel_unique": len(set_excel_num),
        "datamart_unique": len(set_dm_num),
        "intersect_cnt": len(set_excel_num & set_dm_num),
        "only_excel_cnt": len(set_excel_num - set_dm_num),
        "only_datamart_cnt": len(set_dm_num - set_excel_num),
    },
])

display(compare_summary)

## 5) Примеры расхождений (top 100)

In [ ]:
only_excel_inn_df = pd.DataFrame({"inn": sorted(list(set_excel_inn - set_dm_inn))[:100]})
only_dm_inn_df = pd.DataFrame({"inn": sorted(list(set_dm_inn - set_excel_inn))[:100]})

only_excel_agr_df = pd.DataFrame({"agr_id": sorted(list(set_excel_agr - set_dm_agr))[:100]})
only_dm_agr_df = pd.DataFrame({"agr_id": sorted(list(set_dm_agr - set_excel_agr))[:100]})

only_excel_num_df = pd.DataFrame({"c_agr_number": sorted(list(set_excel_num - set_dm_num))[:100]})
only_dm_num_df = pd.DataFrame({"c_agr_number": sorted(list(set_dm_num - set_excel_num))[:100]})

print('Only Excel INN (top 100)')
display(only_excel_inn_df)

print('Only Datamart INN (top 100)')
display(only_dm_inn_df)

print('Only Excel AGR_ID (top 100)')
display(only_excel_agr_df)

print('Only Datamart AGR_ID (top 100)')
display(only_dm_agr_df)

print('Only Excel C_AGR_NUMBER (top 100)')
display(only_excel_num_df)

print('Only Datamart C_AGR_NUMBER (top 100)')
display(only_dm_num_df)

## 6) Доп. контроль: пары INN + AGR_ID

Полезно, если отдельно по INN и AGR все ок, но их сочетания расходятся.

In [ ]:
excel_pair = set(
    excel_norm[["inn", "agr_id"]]
    .dropna(subset=["inn", "agr_id"])
    .apply(tuple, axis=1)
    .tolist()
)
dm_pair = set(
    dm_norm[["inn", "agr_id"]]
    .dropna(subset=["inn", "agr_id"])
    .apply(tuple, axis=1)
    .tolist()
)

pair_summary = pd.DataFrame([
    {"metric": "excel_unique_inn_agr_pairs", "value": len(excel_pair)},
    {"metric": "datamart_unique_inn_agr_pairs", "value": len(dm_pair)},
    {"metric": "intersect_inn_agr_pairs", "value": len(excel_pair & dm_pair)},
    {"metric": "only_excel_inn_agr_pairs", "value": len(excel_pair - dm_pair)},
    {"metric": "only_datamart_inn_agr_pairs", "value": len(dm_pair - excel_pair)},
])
display(pair_summary)

only_excel_pair_df = pd.DataFrame(list(excel_pair - dm_pair), columns=["inn", "agr_id"]).head(100)
only_dm_pair_df = pd.DataFrame(list(dm_pair - excel_pair), columns=["inn", "agr_id"]).head(100)

print('Only Excel INN+AGR pairs (top 100)')
display(only_excel_pair_df)

print('Only Datamart INN+AGR pairs (top 100)')
display(only_dm_pair_df)

## 7) Доп. контроль: пары INN + C_AGR_NUMBER, AGR_ID + C_AGR_NUMBER и тройка INN+AGR_ID+C_AGR_NUMBER

Добавляем недостающие комбинации сравнения, чтобы проверить совпадения по 2 и 3 графам.

In [ ]:
# Пары: INN + C_AGR_NUMBER
excel_pair_inn_num = set(
    excel_norm[["inn", "c_agr_number"]]
    .dropna(subset=["inn", "c_agr_number"])
    .apply(tuple, axis=1)
    .tolist()
)
dm_pair_inn_num = set(
    dm_norm[["inn", "c_agr_number"]]
    .dropna(subset=["inn", "c_agr_number"])
    .apply(tuple, axis=1)
    .tolist()
)

# Пары: AGR_ID + C_AGR_NUMBER
excel_pair_agr_num = set(
    excel_norm[["agr_id", "c_agr_number"]]
    .dropna(subset=["agr_id", "c_agr_number"])
    .apply(tuple, axis=1)
    .tolist()
)
dm_pair_agr_num = set(
    dm_norm[["agr_id", "c_agr_number"]]
    .dropna(subset=["agr_id", "c_agr_number"])
    .apply(tuple, axis=1)
    .tolist()
)

# Тройка: INN + AGR_ID + C_AGR_NUMBER
excel_triplet = set(
    excel_norm[["inn", "agr_id", "c_agr_number"]]
    .dropna(subset=["inn", "agr_id", "c_agr_number"])
    .apply(tuple, axis=1)
    .tolist()
)
dm_triplet = set(
    dm_norm[["inn", "agr_id", "c_agr_number"]]
    .dropna(subset=["inn", "agr_id", "c_agr_number"])
    .apply(tuple, axis=1)
    .tolist()
)

combo_summary = pd.DataFrame([
    {
        "entity": "pair_inn_c_agr_number",
        "excel_unique": len(excel_pair_inn_num),
        "datamart_unique": len(dm_pair_inn_num),
        "intersect_cnt": len(excel_pair_inn_num & dm_pair_inn_num),
        "only_excel_cnt": len(excel_pair_inn_num - dm_pair_inn_num),
        "only_datamart_cnt": len(dm_pair_inn_num - excel_pair_inn_num),
    },
    {
        "entity": "pair_agr_id_c_agr_number",
        "excel_unique": len(excel_pair_agr_num),
        "datamart_unique": len(dm_pair_agr_num),
        "intersect_cnt": len(excel_pair_agr_num & dm_pair_agr_num),
        "only_excel_cnt": len(excel_pair_agr_num - dm_pair_agr_num),
        "only_datamart_cnt": len(dm_pair_agr_num - excel_pair_agr_num),
    },
    {
        "entity": "triplet_inn_agr_id_c_agr_number",
        "excel_unique": len(excel_triplet),
        "datamart_unique": len(dm_triplet),
        "intersect_cnt": len(excel_triplet & dm_triplet),
        "only_excel_cnt": len(excel_triplet - dm_triplet),
        "only_datamart_cnt": len(dm_triplet - excel_triplet),
    },
])

display(combo_summary)

only_excel_inn_num_df = pd.DataFrame(list(excel_pair_inn_num - dm_pair_inn_num), columns=["inn", "c_agr_number"]).head(100)
only_dm_inn_num_df = pd.DataFrame(list(dm_pair_inn_num - excel_pair_inn_num), columns=["inn", "c_agr_number"]).head(100)

only_excel_agr_num_df = pd.DataFrame(list(excel_pair_agr_num - dm_pair_agr_num), columns=["agr_id", "c_agr_number"]).head(100)
only_dm_agr_num_df = pd.DataFrame(list(dm_pair_agr_num - excel_pair_agr_num), columns=["agr_id", "c_agr_number"]).head(100)

only_excel_triplet_df = pd.DataFrame(list(excel_triplet - dm_triplet), columns=["inn", "agr_id", "c_agr_number"]).head(100)
only_dm_triplet_df = pd.DataFrame(list(dm_triplet - excel_triplet), columns=["inn", "agr_id", "c_agr_number"]).head(100)

print('Only Excel INN + C_AGR_NUMBER (top 100)')
display(only_excel_inn_num_df)

print('Only Datamart INN + C_AGR_NUMBER (top 100)')
display(only_dm_inn_num_df)

print('Only Excel AGR_ID + C_AGR_NUMBER (top 100)')
display(only_excel_agr_num_df)

print('Only Datamart AGR_ID + C_AGR_NUMBER (top 100)')
display(only_dm_agr_num_df)

print('Only Excel triplet INN+AGR_ID+C_AGR_NUMBER (top 100)')
display(only_excel_triplet_df)

print('Only Datamart triplet INN+AGR_ID+C_AGR_NUMBER (top 100)')
display(only_dm_triplet_df)